In [15]:
"""
Exponential Smoothing Forecasting Model
========================================
Real-time recursive forecasts of log real TTF NG prices.
α=0.80: s_t = 0.8·r_t + 0.2·s_{t-1}, forecast = exp(s_t) for all h.
"""

import numpy as np
import pandas as pd
import os

# ── Parameters ────────────────────────────────────────────────────────────────
ALPHA        = 0.80           # smoothing parameter (Baumeister et al. 2024).
HORIZONS     = [1, 3, 6, 9, 12, 15, 18, 21, 24]
EVAL_START   = "2006-01-01"   # start of evaluation period
INPUT_FILE   = "Input_TTF_NG_Real_Average_Prices.xlsx"
MODEL_NAME   = "Exponential Smoothing (alpha=0.80)"

In [16]:
# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_excel(INPUT_FILE, parse_dates=["date"])

df = df[["date", "price_real"]].sort_values("date").reset_index(drop=True)

In [17]:
# Log real price
df["log_price"] = np.log(df["price_real"])

# ── Helper: get actual real price for a given year-month ──────────────────────
def get_actual(ym_str):
    """Return the actual real price for a given 'YYYY-MM' string."""
    match = df[df["date"].dt.to_period("M").astype(str) == ym_str]
    if len(match) == 1:
        return match["price_real"].values[0]
    return np.nan

In [ ]:
# ── Main forecasting loop ─────────────────────────────────────────────────────
records = []

# Identify forecast origins: all end-of-month dates from EVAL_START onwards
origins = df[df["date"] >= EVAL_START]["date"].tolist()

for origin_date in origins:

    # All data available up to and including the forecast origin
    history = df[df["date"] <= origin_date]["log_price"].values

    if len(history) < 2:
        continue  # need at least 2 observations to initialise

    # ── Compute recursive exponential smoother up to forecast origin ──────────
    s = history[0]
    for r_t in history[1:]:
        s = ALPHA * r_t + (1 - ALPHA) * s
    # s is now the smoothed log price at the forecast origin

    # ── Generate forecasts for all horizons ───────────────────────────────────
    # ES forecast is flat: same value for all h
    origin_ym = origin_date.strftime("%Y-%m")

    for h in HORIZONS:
        actual_date  = origin_date + pd.DateOffset(months=h)
        actual_ym    = actual_date.strftime("%Y-%m")

        forecast_log   = s                   # flat forecast in log space
        forecast_level = np.exp(forecast_log) # convert back to levels

        actual_val = get_actual(actual_ym)

        records.append({
            "forecast_origin": origin_date.strftime("%Y-%m-%d"),
            "horizon":         h,
            "model":           MODEL_NAME,
            "actual_month":    actual_ym,
            "forecast":        forecast_level,
            "actual":          actual_val
        })

In [13]:
# ── Save output ───────────────────────────────────────────────────────────────
results = pd.DataFrame(records)

results.to_excel("Output_ES_forecasts_long.xlsx", index=False)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Exponential Smoothing forecasts complete.")
print(f"  Alpha:            {ALPHA}")
print(f"  Forecast origins: {results['forecast_origin'].nunique()}")
print(f"  Horizons:         {HORIZONS}")
print(f"  Total rows:       {len(results)}")
print()

Exponential Smoothing forecasts complete.
  Alpha:            0.8
  Forecast origins: 238
  Horizons:         [1, 3, 6, 9, 12, 15, 18, 21, 24]
  Total rows:       2142



In [14]:
# Quick check: first origin
first = results[results["forecast_origin"] == results["forecast_origin"].min()]
print("First forecast origin sample:")
print(first[["forecast_origin","horizon","actual_month","forecast","actual"]].to_string(index=False))

First forecast origin sample:
forecast_origin  horizon actual_month  forecast    actual
     2006-03-31        1      2006-04 29.586938 24.771643
     2006-03-31        3      2006-06 29.586938 22.503165
     2006-03-31        6      2006-09 29.586938 18.320474
     2006-03-31        9      2006-12 29.586938 18.996647
     2006-03-31       12      2007-03 29.586938 11.842205
     2006-03-31       15      2007-06 29.586938 11.939590
     2006-03-31       18      2007-09 29.586938 19.253499
     2006-03-31       21      2007-12 29.586938 25.661291
     2006-03-31       24      2008-03 29.586938 26.245298
